In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
class MOFAVisualizer:
    def __init__(self, factor_data_path, meta_data_path):
        """
        Initializes the visualizer by loading and merging data.
        """
        # Load Data
        self.factors = pd.read_csv(factor_data_path)
        self.meta = pd.read_csv(meta_data_path)

        # Clean metadata (remove index-like columns if they exist)
        if 'Unnamed: 0' in self.meta.columns:
            self.meta = self.meta.drop(columns=['Unnamed: 0'])

        # Merge data on sample_id
        self.df_long = self.factors.melt(id_vars='sample_id', var_name='factor', value_name='factor_value')
        self.merged_df = pd.merge(self.df_long, self.meta, on='sample_id')

        # Set Publication Style
        sns.set_theme(style="ticks", context="paper")
        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['pdf.fonttype'] = 42  # For editable text in Illustrator

    def plot_correlation(self, factors_to_plot, numeric_covariate, ylabel=None, output_path=None):
        """
        Generates scatter plots with regression lines and Pearson correlation.
        """
        subset = self.merged_df[self.merged_df['factor'].isin(factors_to_plot)].copy()

        # 1. Setup Grid with explicit order and independent x-axes
        g = sns.FacetGrid(subset, col="factor", sharey=False, sharex=False,
                          col_order=factors_to_plot, height=4, aspect=1)

        g.map_dataframe(sns.regplot, x='factor_value', y=numeric_covariate, 
                        scatter_kws={'s': 10, 'alpha': 0.5, 'color': 'black'}, 
                        line_kws={'color': '#2c7fb8'})

        # 2. Iterate explicitly over the axes and your ordered list of factors
        #    (We DO NOT use subset.groupby('factor') here, as it auto-sorts alphanumerically)
        for ax, factor_name in zip(g.axes.flat, factors_to_plot):

            # Extract the specific group data for this factor manually
            group = subset[subset['factor'] == factor_name]

            # Drop NAs for calculation
            valid = group.dropna(subset=['factor_value', numeric_covariate])

            if len(valid) > 1:
                r, p = stats.pearsonr(valid['factor_value'], valid[numeric_covariate])
                # print(f"{factor_name}: p={p}") # Debug print

                ax.text(0.05, 0.95, f'r={r:.2f}\np={p:.3f}', transform=ax.transAxes, 
                        fontsize=10, verticalalignment='top', 
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

            # Set Title and Labels
            ax.set_title(f"{factor_name}", fontweight='bold')
            ax.set_xlabel("Factor value")
            if ylabel is not None:
                ax.set_ylabel(ylabel)

        plt.tight_layout()

        if output_path:
            plt.savefig(output_path, dpi=300)
        plt.show()

    def plot_categorical(self, factors_to_plot, categorical_covariate, palette="Set2", output_path=None):
        subset = self.merged_df[self.merged_df['factor'].isin(factors_to_plot)].copy()
        subset = subset.dropna(subset=[categorical_covariate])

        # Figure setup
        plt.figure(figsize=(len(factors_to_plot) * 3.5, 5))
        ax = sns.boxplot(data=subset, x='factor', y='factor_value', hue=categorical_covariate, 
                         palette=palette, fliersize=0, linewidth=1.2, width=0.6)

        sns.stripplot(data=subset, x='factor', y='factor_value', hue=categorical_covariate, 
                      dodge=True, alpha=0.8, palette=palette, size=4, ax=ax, 
                      linewidth=0.8, edgecolor='black', jitter=True, legend=False)

        def get_star_label(p):
            if p < 0.001: return '***'
            if p < 0.01:  return '**'
            if p < 0.05:  return '*'
            return 'n.s.'

        # Add Brackets and Stars
        for i, factor_name in enumerate(factors_to_plot):
            factor_group = subset[subset['factor'] == factor_name]
            groups = [group['factor_value'].values for name, group in factor_group.groupby(categorical_covariate)]

            if len(groups) == 2:
                _, p_val = stats.ttest_ind(groups[0], groups[1], nan_policy='omit')
                print(p_val)
                # Bracket coordinates
                # Since dodge=True, the two boxes are at i - offset and i + offset
                x1, x2 = i - 0.2, i + 0.2 
                y_max = factor_group['factor_value'].max()
                y_range = subset['factor_value'].max() - subset['factor_value'].min()

                # Height of the bracket
                h = y_max + (y_range * 0.05)
                # Height of the "tips" of the bracket
                tip = y_range * 0.02

                # Draw the bracket line: [x1, x1, x2, x2] to [h-tip, h, h, h-tip]
                plt.plot([x1, x1, x2, x2], [h - tip, h, h, h - tip], lw=1.2, c='black')

                # Place the star
                plt.text((x1 + x2) * 0.5, h, get_star_label(p_val), 
                         ha='center', va='bottom', fontsize=12, fontweight='bold')

            # CASE 2: 3+ Groups (ANOVA Global P-value)
            elif len(groups) > 2:
                # Calculate local height variables for the label
                y_max = factor_group['factor_value'].max()
                y_range = subset['factor_value'].max() - subset['factor_value'].min()
                h = y_max + (y_range * 0.05)

                # Perform ANOVA
                f_stat, p_val = stats.f_oneway(*groups)
                print(f"ANOVA p-value for {factor_name}: {p_val}")


                # Format label: Stars + specific p-value
                stat_label = f"ANOVA p={p_val:.3f}"

                # Place the text centered above the current factor
                plt.text(i, h, stat_label, ha='center', va='bottom', 
                         fontsize=10, fontweight='bold', color='black')

        sns.despine()
        plt.title("")
        plt.ylabel("Factor value", fontsize=12)
        plt.xlabel("")

        # Legend inside
        plt.legend(title=categorical_covariate.replace('_', ' ').title(), 
                   loc='lower right', frameon=True)

        plt.tight_layout()
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.show()

# --- Example Usage ---
# viz = MOFAVisualizer('factor_data.csv', 'meta_data.csv')
# viz.plot_correlation(factors_to_plot=['Factor1', 'Factor2'], numeric_covariate='Age', output_path='corr_plot.pdf')
# viz.plot_categorical(factors_to_plot=['Factor1', 'Factor3'], categorical_covariate='Disease_Status')

In [ ]:
viz = MOFAVisualizer('mofa/mofa_workflow/results_mosaic/03_results/03_Factor_Data_v31_norecurr_bulkhvg_MOFA.csv', 'mofa/mofa_workflow/input_data_mosaic/Prepared_Sample_Meta_Data.csv')

In [ ]:
viz.plot_correlation(factors_to_plot=['Factor7'], numeric_covariate='os_years', ylabel = "Overall survival (years)", output_path='figures/Fig3_Fac7_OS.png')

In [ ]:
viz.plot_correlation(factors_to_plot=['Factor7','Factor8','Factor15'], numeric_covariate='pfs_years', ylabel = "Progression free survival (years)" ,output_path='figures/Fig3_Fac15_7_8_PFS_new.png')

In [ ]:
viz.plot_correlation(factors_to_plot=['Factor12'], numeric_covariate='largest_diameter_of_the_primary_tumour_mm_duplicated_0', ylabel="Largest tumor diameter (mm)" ,output_path='figures/Fig3_Fac12_Tumor.png')

In [ ]:
viz.plot_categorical(factors_to_plot=['Factor6', 'Factor10'], categorical_covariate='presence_of_small_cells' , output_path='figures/Fig3_Fac6_10_smallcells.png')

In [ ]:
viz.plot_categorical(factors_to_plot=['Factor10', 'Factor15'], categorical_covariate='alcohol_intake' , output_path='figures/Fig3_Fac10_15_alc.png')

In [ ]:
viz.plot_categorical(factors_to_plot=['Factor5'], categorical_covariate='necrosis' , output_path='figures/Fig3_Fac5_necro.png')

In [ ]:
viz.plot_categorical(factors_to_plot=['Factor15'], categorical_covariate='smoking_status' , output_path='figures/Fig3_Fac15_smoke.png')